In [129]:
# ============================================================
# Mini Trading Dashboard. Near real-time stock price tracker
# Works in: Google Colab, Jupyter Notebook
#
# Libraries required:
#   yfinance, pandas, matplotlib, plotly, requests, time
#
# Notes:
# - yfinance uses Yahoo endpoints. Intraday (1m) often has limits.
# - This implementation avoids threading (Colab-friendly).
# - Robust retry logic, timeouts, and empty-data handling included.
# ============================================================

# ----------------------------
# 1) Install dependencies (Colab)
# ----------------------------
# In Google Colab, uncomment the next line:
# !pip -q install yfinance plotly

import time
import math
import requests
import pandas as pd
import yfinance as yf

import matplotlib.pyplot as plt  # included per requirement (not strictly needed for Plotly charts)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Notebook output helpers
try:
    from IPython.display import display, clear_output
except Exception:
    display = print
    def clear_output(wait=False):  # fallback
        return


In [130]:
# ----------------------------
# 2) User configuration
# ----------------------------
CONFIG = {
    "tickers": ["AAPL", "MSFT", "SPY", "NVDA"],
    "interval": "5m",          # supported: 1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 1d ...
    "period": "1d",            # for intraday dashboards: "1d" or "5d"
    "refresh_seconds": 60,     # refresh cycle
    "max_cycles": None,        # None means run until interrupted. Set e.g. 50 for finite loops.
    "timezone": None,          # None keeps what yfinance returns. You can set "Europe/Rome" if needed.

    # Data fetching reliability
    "request_timeout_seconds": 10,
    "retries": 3,
    "retry_backoff_seconds": 1.5,  # exponential backoff factor

    # Indicators params
    "ma_fast": 20,
    "ma_slow": 50,
    "rsi_period": 14,
    "boll_period": 20,
    "boll_std": 2.0,

    # Alerts (set to None to disable specific alert types)
    "alerts": {
        # price threshold: {"AAPL": {"above": 200, "below": 170}, "SPY": {"above": 550}}
        "price_thresholds": {},

        # daily change threshold in percent, e.g. 2.5 triggers if abs(change) >= 2.5
        "daily_change_pct": 3.0,

        # RSI thresholds
        "rsi_overbought": 70,
        "rsi_oversold": 30,
    },

    # Plot controls
    "show_multi_ticker_comparison": True,
    "comparison_normalize_to_100": True,  # normalize each series to start at 100
    "max_points_per_ticker": 1200,        # guardrail for heavy intraday data
}

In [131]:
# ----------------------------
# 3) Small utilities
# ----------------------------
def _now_ts():
    # Use pandas Timestamp for consistent printing
    return pd.Timestamp.utcnow()

def _safe_float(x):
    try:
        if x is None:
            return math.nan
        return float(x)
    except Exception:
        return math.nan

def _clamp_df_points(df: pd.DataFrame, max_points: int) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    if max_points is None:
        return df
    if len(df) > max_points:
        return df.iloc[-max_points:].copy()
    return df

def _validate_interval(interval: str) -> None:
    allowed = {"1m", "2m", "5m", "15m", "30m", "60m", "90m", "1h", "1d"}
    if interval not in allowed:
        raise ValueError(f"Unsupported interval='{interval}'. Try one of: {sorted(allowed)}")

In [132]:
# ----------------------------
# 4) Indicators
# ----------------------------
def compute_moving_averages(df: pd.DataFrame, ma_fast: int = 20, ma_slow: int = 50) -> pd.DataFrame:
    out = df.copy()
    if "Close" not in out.columns or out.empty:
        out["MA_FAST"] = pd.Series(dtype="float64")
        out["MA_SLOW"] = pd.Series(dtype="float64")
        return out
    out["MA_FAST"] = out["Close"].rolling(window=ma_fast, min_periods=max(2, ma_fast // 4)).mean()
    out["MA_SLOW"] = out["Close"].rolling(window=ma_slow, min_periods=max(2, ma_slow // 4)).mean()
    return out

def compute_rsi(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    out = df.copy()
    if "Close" not in out.columns or out.empty:
        out["RSI"] = pd.Series(dtype="float64")
        return out

    delta = out["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # Wilder's smoothing via EMA with alpha=1/period is a common approach
    avg_gain = gain.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False, min_periods=period).mean()

    rs = avg_gain / avg_loss.replace(0, pd.NA)
    out["RSI"] = 100 - (100 / (1 + rs.astype("float64")))
    return out

def compute_bollinger_bands(df: pd.DataFrame, period: int = 20, n_std: float = 2.0) -> pd.DataFrame:
    out = df.copy()
    if "Close" not in out.columns or out.empty:
        out["BB_MID"] = pd.Series(dtype="float64")
        out["BB_UPPER"] = pd.Series(dtype="float64")
        out["BB_LOWER"] = pd.Series(dtype="float64")
        return out

    mid = out["Close"].rolling(window=period, min_periods=max(2, period // 4)).mean()
    std = out["Close"].rolling(window=period, min_periods=max(2, period // 4)).std()

    out["BB_MID"] = mid
    out["BB_UPPER"] = mid + n_std * std
    out["BB_LOWER"] = mid - n_std * std
    return out

def compute_indicators(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    out = df.copy()
    out = compute_moving_averages(out, cfg["ma_fast"], cfg["ma_slow"])
    out = compute_rsi(out, cfg["rsi_period"])
    out = compute_bollinger_bands(out, cfg["boll_period"], cfg["boll_std"])
    return out


In [133]:
# ----------------------------
# 5) Data fetching with retries, timeouts
# ----------------------------
import time
import requests
import pandas as pd
import yfinance as yf

def _preflight_internet(timeout_seconds: int = 5) -> bool:
    """
    Optional sanity check. Helps fail fast when the runtime has no connectivity.
    Uses requests as required by spec.
    """
    try:
        r = requests.get("https://www.google.com", timeout=timeout_seconds)
        return r.status_code == 200
    except Exception:
        return False


def _standardize_ohlcv_columns(df: pd.DataFrame, ticker: str | None = None) -> pd.DataFrame:
    """
    Ensure a flat OHLCV dataframe with columns:
    Open, High, Low, Close, Adj Close, Volume.

    Handles yfinance outputs:
    - Single ticker: flat columns already.
    - Multi ticker: MultiIndex columns can be (ticker, field) or (field, ticker).
    """
    if df is None or df.empty:
        return df

    if isinstance(df.columns, pd.MultiIndex):
        lvl0 = set(df.columns.get_level_values(0))
        lvl1 = set(df.columns.get_level_values(1))
        ohlcv = {"Open", "High", "Low", "Close", "Adj Close", "Volume"}

        # Layout (field, ticker)
        if len(lvl0 & ohlcv) > 0:
            if ticker is None:
                ticker = df.columns.get_level_values(1)[0]
            df = df.xs(ticker, axis=1, level=1, drop_level=True)
        # Layout (ticker, field)
        else:
            if ticker is None:
                ticker = df.columns.get_level_values(0)[0]
            df = df.xs(ticker, axis=1, level=0, drop_level=True)

    # Make sure index is datetime and sorted
    df.index = pd.to_datetime(df.index, errors="coerce")
    df = df[~df.index.isna()].sort_index()

    return df


def fetch_market_data(
    tickers: list[str],
    interval: str = "1m",
    period: str = "1d",
    request_timeout_seconds: int = 10,
    retries: int = 3,
    retry_backoff_seconds: float = 1.5,
    max_points_per_ticker: int | None = 1200,
    preflight_check: bool = False,
) -> dict[str, pd.DataFrame]:
    """
    Fetch OHLCV data for multiple tickers using yfinance.
    Returns: {ticker: DataFrame(index=datetime, columns=Open/High/Low/Close/Adj Close/Volume)}

    Notes on timeouts:
    - yfinance uses requests internally and does not reliably expose a per-call timeout.
    - request_timeout_seconds is used for optional preflight connectivity checking.
    """
    _validate_interval(interval)

    tickers = [str(t).strip().upper() for t in tickers if str(t).strip()]
    out: dict[str, pd.DataFrame] = {t: pd.DataFrame() for t in tickers}

    if not tickers:
        out["_ERROR"] = pd.DataFrame({"message": ["No tickers provided."]})
        return out

    if preflight_check and not _preflight_internet(timeout_seconds=request_timeout_seconds):
        out["_ERROR"] = pd.DataFrame({"message": ["No internet connectivity (preflight failed)."]})
        return out

    last_err: Exception | None = None

    for attempt in range(1, retries + 1):
        try:
            raw = yf.download(
                tickers=tickers,
                period=period,
                interval=interval,
                group_by="ticker",
                auto_adjust=False,
                threads=False,
                progress=False,
            )

            # If raw is empty, return empties (no exception thrown by yfinance)
            if raw is None or (isinstance(raw, pd.DataFrame) and raw.empty):
                return out

            # Case 1: Single ticker, flat columns
            if isinstance(raw, pd.DataFrame) and isinstance(raw.columns, pd.Index) and not isinstance(raw.columns, pd.MultiIndex):
                t = tickers[0]
                df = _standardize_ohlcv_columns(raw.copy(), t)
                if not df.empty:
                    df = _clamp_df_points(df, max_points_per_ticker)
                out[t] = df
                return out

            # Case 2: Multi ticker, MultiIndex columns
            # Handle both possible layouts using the standardizer on the full raw DF.
            available = set(raw.columns.get_level_values(0)) if isinstance(raw.columns, pd.MultiIndex) else set()

            for t in tickers:
                df_t = pd.DataFrame()

                # Preferred: exact match when layout is (ticker, field)
                if isinstance(raw.columns, pd.MultiIndex) and t in available:
                    try:
                        df_t = raw[t].copy()
                    except Exception:
                        df_t = pd.DataFrame()

                # Best-effort match if symbol got normalized (dots/hyphens)
                if df_t.empty and isinstance(raw.columns, pd.MultiIndex):
                    matches = [lvl for lvl in set(raw.columns.get_level_values(0)) if str(lvl).strip().upper() == t.upper()]
                    if matches:
                        try:
                            df_t = raw[matches[0]].copy()
                        except Exception:
                            df_t = pd.DataFrame()

                # If we still have MultiIndex columns (or got the other layout), standardize via helper
                if not df_t.empty:
                    df_t = _standardize_ohlcv_columns(df_t, t)
                else:
                    # Try extracting from the full raw dataframe in case layout is (field, ticker)
                    try:
                        df_t2 = _standardize_ohlcv_columns(raw.copy(), t)
                        df_t = df_t2 if isinstance(df_t2, pd.DataFrame) else pd.DataFrame()
                    except Exception:
                        df_t = pd.DataFrame()

                if not df_t.empty:
                    df_t = _clamp_df_points(df_t, max_points_per_ticker)

                out[t] = df_t

            return out

        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(retry_backoff_seconds ** (attempt - 1))
            else:
                break

    # Exhausted retries
    if last_err is not None:
        out["_ERROR"] = pd.DataFrame({"message": [str(last_err)]})
    else:
        out["_ERROR"] = pd.DataFrame({"message": ["Unknown error while fetching market data."]})
    return out

In [134]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def build_dashboard_figure(
    data: dict[str, pd.DataFrame],
    focus_ticker: str,
    interval: str,
    period: str
) -> go.Figure:
    tickers = [t for t in data.keys() if not str(t).startswith("_")]
    focus_ticker = str(focus_ticker).upper().strip()

    # More breathing room. Titles will live in margins, not inside plots.
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.28,        # more space between panels
        row_heights=[0.46, 0.54],
        specs=[[{"secondary_y": False}], [{"secondary_y": True}]],
        subplot_titles=None,
    )

    # ----------------------------
    # 1) Multi-ticker normalized
    # ----------------------------
    for t in tickers:
        df = data.get(t)
        if df is None or df.empty or "Close" not in df.columns:
            continue

        s = df["Close"].ffill().dropna()
        if len(s) < 2:
            continue

        norm = (s / s.iloc[0]) * 100.0
        fig.add_trace(
            go.Scatter(
                x=norm.index,
                y=norm.values,
                mode="lines",
                name=t,
                legendgroup="tickers",
                hovertemplate=f"{t}<br>%{{x}}<br>Index=%{{y:.2f}}<extra></extra>",
            ),
            row=1,
            col=1,
        )

    fig.update_yaxes(title_text="Index", row=1, col=1)

    # ----------------------------
    # 2) Focus ticker details
    # ----------------------------
    df_f = data.get(focus_ticker)
    if df_f is not None and not df_f.empty and "Close" in df_f.columns:
        df_f = df_f.sort_index()
        close = df_f["Close"].ffill()

        # Price traces in legend
        fig.add_trace(
            go.Scatter(
                x=close.index,
                y=close.values,
                mode="lines",
                name="Close",
                legendgroup="price",
                hovertemplate="%{x}<br>Close=%{y:.2f}<extra></extra>",
            ),
            row=2,
            col=1,
            secondary_y=False,
        )

        if "MA20" in df_f.columns:
            ma20 = df_f["MA20"].ffill()
            fig.add_trace(
                go.Scatter(
                    x=ma20.index,
                    y=ma20.values,
                    mode="lines",
                    name="MA20",
                    legendgroup="price",
                    hovertemplate="%{x}<br>MA20=%{y:.2f}<extra></extra>",
                ),
                row=2,
                col=1,
                secondary_y=False,
            )

        if "MA50" in df_f.columns:
            ma50 = df_f["MA50"].ffill()
            fig.add_trace(
                go.Scatter(
                    x=ma50.index,
                    y=ma50.values,
                    mode="lines",
                    name="MA50",
                    legendgroup="price",
                    hovertemplate="%{x}<br>MA50=%{y:.2f}<extra></extra>",
                ),
                row=2,
                col=1,
                secondary_y=False,
            )

        # Bollinger. Keep out of legend to avoid clutter
        if "BB_upper" in df_f.columns and "BB_lower" in df_f.columns:
            bb_u = df_f["BB_upper"].ffill()
            bb_l = df_f["BB_lower"].ffill()

            fig.add_trace(
                go.Scatter(
                    x=bb_u.index,
                    y=bb_u.values,
                    mode="lines",
                    name="BB Upper",
                    showlegend=False,
                    hovertemplate="%{x}<br>BBU=%{y:.2f}<extra></extra>",
                ),
                row=2,
                col=1,
                secondary_y=False,
            )
            fig.add_trace(
                go.Scatter(
                    x=bb_l.index,
                    y=bb_l.values,
                    mode="lines",
                    name="BB Lower",
                    showlegend=False,
                    fill="tonexty",
                    hovertemplate="%{x}<br>BBL=%{y:.2f}<extra></extra>",
                ),
                row=2,
                col=1,
                secondary_y=False,
            )

        # RSI + Volume out of legend
        if "RSI" in df_f.columns:
            rsi = df_f["RSI"].ffill()
            fig.add_trace(
                go.Scatter(
                    x=rsi.index,
                    y=rsi.values,
                    mode="lines",
                    name="RSI",
                    showlegend=False,
                    hovertemplate="%{x}<br>RSI=%{y:.1f}<extra></extra>",
                ),
                row=2,
                col=1,
                secondary_y=True,
            )
            fig.update_yaxes(title_text="RSI", range=[0, 100], row=2, col=1, secondary_y=True)

        if "Volume" in df_f.columns:
            vol = df_f["Volume"].fillna(0)
            fig.add_trace(
                go.Bar(
                    x=vol.index,
                    y=vol.values,
                    name="Volume",
                    showlegend=False,
                    opacity=0.25,
                    hovertemplate="%{x}<br>Vol=%{y:,.0f}<extra></extra>",
                ),
                row=2,
                col=1,
                secondary_y=True,
            )

        fig.update_yaxes(title_text="Price", row=2, col=1, secondary_y=False)

    # ----------------------------
    # HARD anti-overlap layout
    # Key changes:
    # - Huge top margin
    # - Legend placed well above plots
    # - Titles placed in the margin (paper coords)
    # ----------------------------
    fig.update_layout(
        height=900,
        margin=dict(l=50, r=30, t=240, b=70),  # IMPORTANT: big top margin
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.18,           # IMPORTANT: legend clearly in the top margin
            xanchor="left",
            x=0.0,
            font=dict(size=10),
            itemclick="toggleothers",
            itemdoubleclick="toggle",
        ),
        hovermode="x unified",
    )

    # Titles in the top margin. These will never overlap any subplot.
    fig.add_annotation(
        text="Multi-ticker comparison (normalized to 100)",
        xref="paper",
        yref="paper",
        x=0.0,
        y=1.13,
        xanchor="left",
        yanchor="bottom",
        showarrow=False,
        font=dict(size=14),
    )

    fig.add_annotation(
        text=f"{focus_ticker} . interval={interval} . period={period}",
        xref="paper",
        yref="paper",
        x=0.0,
        y=0.60,          # safely above row 2 plot area thanks to vertical_spacing + big top margin
        xanchor="left",
        yanchor="bottom",
        showarrow=False,
        font=dict(size=14),
    )

    return fig

In [135]:
# ----------------------------
# 7) Alerts
# ----------------------------
def _crossed_threshold(prev_val: float, curr_val: float, threshold: float, direction: str) -> bool:
    if math.isnan(prev_val) or math.isnan(curr_val) or threshold is None:
        return False
    if direction == "above":
        return prev_val < threshold <= curr_val
    if direction == "below":
        return prev_val > threshold >= curr_val
    return False

def evaluate_alerts(data_by_ticker: dict[str, pd.DataFrame], cfg: dict, prev_snapshot: dict[str, dict] | None) -> list[str]:
    """
    Returns list of alert messages.
    prev_snapshot: per ticker values from the previous cycle (for crossing detection).
    """
    alerts_cfg = cfg.get("alerts", {}) or {}
    price_thresholds = alerts_cfg.get("price_thresholds", {}) or {}
    daily_change_pct_thr = alerts_cfg.get("daily_change_pct", None)
    rsi_ob = alerts_cfg.get("rsi_overbought", None)
    rsi_os = alerts_cfg.get("rsi_oversold", None)

    messages: list[str] = []

    for t, df in data_by_ticker.items():
        if t == "_ERROR":
            continue
        if df is None or df.empty:
            continue
        if "Close" not in df.columns:
            continue

        curr_price = _safe_float(df["Close"].iloc[-1])
        prev_price = math.nan
        if prev_snapshot and t in prev_snapshot:
            prev_price = _safe_float(prev_snapshot[t].get("price"))

        # Price threshold crossings
        thr = price_thresholds.get(t, {}) if isinstance(price_thresholds, dict) else {}
        if isinstance(thr, dict):
            if "above" in thr:
                if _crossed_threshold(prev_price, curr_price, _safe_float(thr["above"]), "above"):
                    messages.append(f"[PRICE] {t} crossed ABOVE {thr['above']} (now {curr_price:.4f})")
            if "below" in thr:
                if _crossed_threshold(prev_price, curr_price, _safe_float(thr["below"]), "below"):
                    messages.append(f"[PRICE] {t} crossed BELOW {thr['below']} (now {curr_price:.4f})")

        # Daily change threshold
        first_close = _safe_float(df["Close"].iloc[0])
        daily_pct = math.nan
        if not (math.isnan(first_close) or first_close == 0 or math.isnan(curr_price)):
            daily_pct = (curr_price / first_close - 1.0) * 100.0
        if daily_change_pct_thr is not None and not math.isnan(daily_pct):
            if abs(daily_pct) >= float(daily_change_pct_thr):
                messages.append(f"[MOVE] {t} daily move {daily_pct:+.2f}% exceeds {daily_change_pct_thr}%")

        # RSI alerts, requires RSI present
        if "RSI" in df.columns and not df["RSI"].empty:
            curr_rsi = _safe_float(df["RSI"].iloc[-1])
            if rsi_ob is not None and not math.isnan(curr_rsi) and curr_rsi >= float(rsi_ob):
                messages.append(f"[RSI] {t} RSI {curr_rsi:.1f} overbought (>= {rsi_ob})")
            if rsi_os is not None and not math.isnan(curr_rsi) and curr_rsi <= float(rsi_os):
                messages.append(f"[RSI] {t} RSI {curr_rsi:.1f} oversold (<= {rsi_os})")

    return messages

def build_prev_snapshot(data_by_ticker: dict[str, pd.DataFrame]) -> dict[str, dict]:
    snap: dict[str, dict] = {}
    for t, df in data_by_ticker.items():
        if t == "_ERROR":
            continue
        if df is None or df.empty or "Close" not in df.columns:
            continue
        snap[t] = {
            "price": _safe_float(df["Close"].iloc[-1]),
        }
    return snap


In [136]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_price_chart(
    ticker: str,
    df: pd.DataFrame,
    cfg: dict,
) -> go.Figure:
    """
    Interactive chart with:
    - Price (line)
    - MA20/MA50 (lines)
    - Bollinger Bands (lines + filled area)
    - Volume (bars)
    """
    if df is None or df.empty:
        fig = go.Figure()
        fig.update_layout(title=f"{ticker} (no data)")
        return fig

    # Ensure 'Close' column exists before trying to compute indicators or plot price
    if "Close" not in df.columns:
        fig = go.Figure()
        fig.update_layout(title=f"{ticker} (no 'Close' price data available)", height=300)
        return fig

    # Now that we know 'Close' exists, compute indicators
    df = df.copy()
    needed_cols = {"MA_FAST", "MA_SLOW", "BB_UPPER", "BB_LOWER", "BB_MID", "RSI"}
    if not needed_cols.issubset(set(df.columns)):
        df = compute_indicators(df, cfg)

    x = df.index

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.55, 0.25, 0.20],
        specs=[[{"type": "scatter"}], [{"type": "bar"}], [{"type": "scatter"}]],
        subplot_titles=[f"{ticker} Price & Moving Averages", "Volume", "RSI"]
    )

    # Price
    fig.add_trace(go.Scatter(x=x, y=df["Close"], mode="lines", name="Close"), row=1, col=1)

    # MAs
    if "MA_FAST" in df.columns:
        fig.add_trace(go.Scatter(x=x, y=df["MA_FAST"], mode="lines", name=f"MA{cfg['ma_fast']}"), row=1, col=1)
    if "MA_SLOW" in df.columns:
        fig.add_trace(go.Scatter(x=x, y=df["MA_SLOW"], mode="lines", name=f"MA{cfg['ma_slow']}"), row=1, col=1)

    # Bollinger Bands
    if "BB_UPPER" in df.columns and "BB_LOWER" in df.columns:
        fig.add_trace(go.Scatter(x=x, y=df["BB_UPPER"], mode="lines", name="BB Upper", line=dict(width=1)), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=x, y=df["BB_LOWER"], mode="lines", name="BB Lower",
            line=dict(width=1),
            fill="tonexty",  # fill between upper and lower (requires ordering)
        ), row=1, col=1)

    # Volume
    if "Volume" in df.columns:
        fig.add_trace(go.Bar(x=x, y=df["Volume"], name="Volume"), row=2, col=1)

    # RSI
    if "RSI" in df.columns:
        fig.add_trace(go.Scatter(x=x, y=df["RSI"], mode="lines", name="RSI"), row=3, col=1)
        # RSI guides
        fig.add_hline(y=cfg["alerts"].get("rsi_overbought", 70), row=3, col=1)
        fig.add_hline(y=cfg["alerts"].get("rsi_oversold", 30), row=3, col=1)

    # Layout adjustments to prevent overlap
    fig.update_layout(
        title=f"{ticker} . interval={cfg['interval']} . period={cfg['period']}",
        height=900,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0), # Legend slightly above plot area
        margin=dict(l=40, r=20, t=80, b=40), # Increased top margin for title
    )
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)
    fig.update_yaxes(title_text="RSI", row=3, col=1, range=[0, 100])

    return fig

def plot_multi_ticker_comparison(data_by_ticker: dict[str, pd.DataFrame], cfg: dict) -> go.Figure:
    """
    Multi-ticker comparison chart.
    Optionally normalizes each series to start at 100.
    """
    fig = go.Figure()
    series_added = 0

    for t, df in data_by_ticker.items():
        if t == "_ERROR":
            continue
        if df is None or df.empty or "Close" not in df.columns:
            continue

        s = df["Close"].copy()
        s = s.dropna()
        if s.empty:
            continue

        if cfg.get("comparison_normalize_to_100", True):
            base = float(s.iloc[0])
            if base != 0:
                s = (s / base) * 100.0

        fig.add_trace(go.Scatter(x=s.index, y=s.values, mode="lines", name=t))
        series_added += 1

    title = "Multi-ticker comparison"
    if cfg.get("comparison_normalize_to_100", True):
        title += " (normalized to 100)"
    if series_added == 0:
        title += " . no data"

    # Layout adjustments to prevent overlap
    fig.update_layout(
        title=title,
        height=450,
        margin=dict(l=40, r=20, t=80, b=40), # Increased top margin for title
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0), # Legend slightly above plot area
    )
    fig.update_yaxes(title_text="Index" if cfg.get("comparison_normalize_to_100", True) else "Price")
    return fig

In [137]:
# ----------------------------
# 9) Main live loop
# ----------------------------
def run_live_tracker(cfg: dict):
    """
    Main entry point.
    Clears notebook output each cycle and redraws:
    - Dashboard table
    - Multi-ticker chart
    - Per-ticker detailed chart
    - Alerts
    """
    tickers = cfg["tickers"]
    _validate_interval(cfg["interval"])

    # Optional connectivity preflight
    if not _preflight_internet(timeout_seconds=min(5, cfg.get("request_timeout_seconds", 10))):
        print("Warning: internet preflight failed. yfinance requests may fail in this runtime.\n")

    prev_snapshot = None
    cycle = 0

    while True:
        if cfg.get("max_cycles") is not None and cycle >= int(cfg["max_cycles"]):
            print("Reached max_cycles. Stopping.")
            break

        cycle += 1
        last_update = _now_ts()

        data_by_ticker = fetch_market_data(
            tickers=tickers,
            interval=cfg["interval"],
            period=cfg["period"],
            request_timeout_seconds=cfg["request_timeout_seconds"],
            retries=cfg["retries"],
            retry_backoff_seconds=cfg["retry_backoff_seconds"],
            max_points_per_ticker=cfg.get("max_points_per_ticker", 1200),
        )

        # Compute indicators per ticker (safe even if empty)
        for t in tickers:
            df = data_by_ticker.get(t, pd.DataFrame())
            if df is None or df.empty:
                continue
            data_by_ticker[t] = compute_indicators(df, cfg)

        # Alerts
        alert_messages = evaluate_alerts(data_by_ticker, cfg, prev_snapshot)
        prev_snapshot = build_prev_snapshot(data_by_ticker)

        # Build table
        table = build_dashboard_table(data_by_ticker, last_update_ts=last_update)

        # Render
        clear_output(wait=True)

        print(f"Mini Trading Dashboard . cycle={cycle} . refreshed @ {last_update} UTC")
        if "_ERROR" in data_by_ticker and not data_by_ticker["_ERROR"].empty:
            print(f"Fetch error (last attempt): {data_by_ticker['_ERROR']['message'].iloc[0]}\n")

        # Display the table nicely
        if not table.empty:
            display(table.style.format({
                "Last price": "{:.4f}",
                "Daily % change": "{:+.2f}%",
                "Volume": "{:,.0f}",
            }, na_rep="—"))
        else:
            print("No table data.\n")

        # Alerts panel
        if alert_messages:
            print("\nALERTS:")
            for msg in alert_messages:
                print(" -", msg)
        else:
            print("\nALERTS: none")

        # Multi ticker comparison
        if cfg.get("show_multi_ticker_comparison", True):
            fig_comp = plot_multi_ticker_comparison(data_by_ticker, cfg)
            fig_comp.show()

        # Detailed per ticker
        for t in tickers:
            fig = plot_price_chart(t, data_by_ticker.get(t, pd.DataFrame()), cfg)
            fig.show()

        # Sleep. Keep it simple, no threads.
        # If yfinance is slow, the cycle time naturally increases. This avoids overlapping calls.
        time.sleep(max(1, int(cfg["refresh_seconds"])))


In [138]:
# ----------------------------
# 10) How to run
# ----------------------------
# In Colab / Jupyter:
# 1) Run this whole cell (or notebook).
# 2) Optionally edit CONFIG.
# 3) Run:
#    run_live_tracker(CONFIG)
#
# To stop: Interrupt the kernel (Colab: Runtime -> Interrupt execution)

# Example: enable a price alert for AAPL above 200
# CONFIG["alerts"]["price_thresholds"] = {"AAPL": {"above": 200}}

# Start the dashboard
# run_live_tracker(CONFIG)

### Start the Live Dashboard

To begin tracking, execute the cell below. Remember that you can modify the `CONFIG` dictionary in section 2 beforehand to customize the tickers, refresh rate, and other settings.

**To stop the dashboard, go to `Runtime` -> `Interrupt execution` or press `Ctrl+C`.**

In [139]:
print("---- Debugging Data Fetch ----")

debug_ticker = str(CONFIG["tickers"][0]).upper().strip()
interval = CONFIG["interval"]
period = CONFIG["period"]

print(f"Ticker: {debug_ticker}")
print(f"Interval: {interval} . Period: {period}")

debug_data = fetch_market_data(
    tickers=[debug_ticker],
    interval=interval,
    period=period,
    request_timeout_seconds=CONFIG.get("request_timeout_seconds", 10),
    retries=CONFIG.get("retries", 3),
    retry_backoff_seconds=CONFIG.get("retry_backoff_seconds", 1.5),
    max_points_per_ticker=CONFIG.get("max_points_per_ticker", 1200),
)

# 1) Handle fetch errors
if "_ERROR" in debug_data and isinstance(debug_data["_ERROR"], pd.DataFrame) and not debug_data["_ERROR"].empty:
    msg = debug_data["_ERROR"].get("message", pd.Series(["Unknown error"])).iloc[0]
    print(f"[FETCH ERROR] {debug_ticker}: {msg}")

else:
    df_raw = debug_data.get(debug_ticker)

    # 2) Validate raw dataframe
    if df_raw is None:
        print(f"[NO DATA] Missing key in returned dict for ticker: {debug_ticker}")
        print(f"Available keys: {list(debug_data.keys())}")

    elif not isinstance(df_raw, pd.DataFrame):
        print(f"[BAD TYPE] Expected DataFrame, got: {type(df_raw)}")

    elif df_raw.empty:
        print(f"[EMPTY DF] No rows returned for {debug_ticker}.")
        print("Try: interval='5m' or '15m' . Or period='5d'.")

    else:
        # 3) Quick raw summary
        print("\n--- Raw DF Summary ---")
        print(f"Rows: {len(df_raw):,} . Cols: {df_raw.shape[1]}")
        print(f"Index type: {type(df_raw.index)}")
        print(f"Index min: {df_raw.index.min()} . max: {df_raw.index.max()}")
        print(f"Columns: {list(df_raw.columns)}")

        expected = ["Open", "High", "Low", "Close", "Volume"]
        missing = [c for c in expected if c not in df_raw.columns]
        if missing:
            print(f"[WARN] Missing expected columns: {missing}")

        # 4) Show last values (if present)
        if "Close" in df_raw.columns:
            last_close = df_raw["Close"].ffill().iloc[-1]
            print(f"Last Close: {last_close:.4f}")

        if "Volume" in df_raw.columns:
            last_vol = df_raw["Volume"].fillna(0).iloc[-1]
            try:
                last_vol = int(last_vol)
            except Exception:
                pass
            try:
                print(f"Last Volume: {last_vol:,}")
            except Exception:
                print(f"Last Volume: {last_vol}")

        # 5) Preview (small, readable)
        print("\n--- Raw Head (3) ---")
        display(df_raw.head(3))
        print("\n--- Raw Tail (3) ---")
        display(df_raw.tail(3))

        # 6) Compute indicators with guard
        print("\n---- Computing indicators ----")
        try:
            df_processed = compute_indicators(df_raw, CONFIG)
        except Exception as e:
            print(f"[INDICATORS ERROR] {type(e).__name__}: {e}")
            raise

        # 7) Processed summary
        print("\n--- Processed DF Summary ---")
        print(f"Rows: {len(df_processed):,} . Cols: {df_processed.shape[1]}")
        print(f"Index min: {df_processed.index.min()} . max: {df_processed.index.max()}")
        print(f"Columns: {list(df_processed.columns)}")

        # 8) Identify indicator columns (new vs raw)
        raw_cols = set(df_raw.columns)
        new_cols = [c for c in df_processed.columns if c not in raw_cols]

        print("\n--- Indicator columns (new vs raw) ---")
        print(f"Count: {len(new_cols)}")
        for c in new_cols:
            print(f"- {c}")

        # 9) NaN ratio on indicators
        if new_cols:
            nan_ratio = df_processed[new_cols].isna().mean().sort_values(ascending=False)
            print("\n--- NaN ratio (top 15 indicators) ---")
            display(nan_ratio.head(15))

        # 10) Preview processed
        print("\n--- Processed Head (3) ---")
        display(df_processed.head(3))
        print("\n--- Processed Tail (3) ---")
        display(df_processed.tail(3))



---- Debugging Data Fetch ----
Ticker: AAPL
Interval: 5m . Period: 1d

--- Raw DF Summary ---
Rows: 78 . Cols: 6
Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
Index min: 2026-03-04 14:30:00+00:00 . max: 2026-03-04 20:55:00+00:00
Columns: ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
Last Close: 262.4500
Last Volume: 2,637,679

--- Raw Head (3) ---


Price,Open,High,Low,Close,Adj Close,Volume
Datetime,,,,,,
2026-03-04 14:30:00+00:00,264.649994,266.149994,263.920013,264.679901,264.679901,2172518
2026-03-04 14:35:00+00:00,264.635010,264.753510,262.725006,263.225006,263.225006,588778
2026-03-04 14:40:00+00:00,263.231110,263.231110,262.140015,262.299988,262.299988,549218



--- Raw Tail (3) ---


Price,Open,High,Low,Close,Adj Close,Volume
Datetime,,,,,,
2026-03-04 20:45:00+00:00,262.339996,262.450012,262.179993,262.234985,262.234985,308315
2026-03-04 20:50:00+00:00,262.230011,263.160004,262.164612,262.859985,262.859985,1044816
2026-03-04 20:55:00+00:00,262.850006,263.119995,262.445007,262.450012,262.450012,2637679



---- Computing indicators ----

--- Processed DF Summary ---
Rows: 78 . Cols: 12
Index min: 2026-03-04 14:30:00+00:00 . max: 2026-03-04 20:55:00+00:00
Columns: ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'MA_FAST', 'MA_SLOW', 'RSI', 'BB_MID', 'BB_UPPER', 'BB_LOWER']

--- Indicator columns (new vs raw) ---
Count: 6
- MA_FAST
- MA_SLOW
- RSI
- BB_MID
- BB_UPPER
- BB_LOWER

--- NaN ratio (top 15 indicators) ---


,0
Price,
RSI,0.179487
MA_SLOW,0.141026
MA_FAST,0.051282
BB_MID,0.051282
BB_UPPER,0.051282
BB_LOWER,0.051282



--- Processed Head (3) ---


Price,Open,High,Low,Close,Adj Close,Volume,MA_FAST,MA_SLOW,RSI,BB_MID,BB_UPPER,BB_LOWER
Datetime,,,,,,,,,,,,
2026-03-04 14:30:00+00:00,264.649994,266.149994,263.920013,264.679901,264.679901,2172518,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-04 14:35:00+00:00,264.635010,264.753510,262.725006,263.225006,263.225006,588778,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-04 14:40:00+00:00,263.231110,263.231110,262.140015,262.299988,262.299988,549218,NaN,NaN,NaN,NaN,NaN,NaN



--- Processed Tail (3) ---


Price,Open,High,Low,Close,Adj Close,Volume,MA_FAST,MA_SLOW,RSI,BB_MID,BB_UPPER,BB_LOWER
Datetime,,,,,,,,,,,,
2026-03-04 20:45:00+00:00,262.339996,262.450012,262.179993,262.234985,262.234985,308315,262.99832,263.838809,29.615342,262.99832,263.866624,262.130016
2026-03-04 20:50:00+00:00,262.230011,263.160004,262.164612,262.859985,262.859985,1044816,262.96832,263.789409,44.668045,262.96832,263.810531,262.126108
2026-03-04 20:55:00+00:00,262.850006,263.119995,262.445007,262.450012,262.450012,2637679,262.91732,263.733610,38.805457,262.91732,263.755138,262.079503


In [ ]:
run_live_tracker(CONFIG)

Mini Trading Dashboard . cycle=2 . refreshed @ 2026-03-05 09:11:05.457102+00:00 UTC


,Ticker,Last price,Daily % change,Volume,Last update (UTC)
0,AAPL,262.4500,-0.84%,"2,637,679",2026-03-05 09:11:05.457102+00:00
1,MSFT,405.0000,+0.07%,"1,764,328",2026-03-05 09:11:05.457102+00:00
2,NVDA,182.9100,+0.40%,"6,350,086",2026-03-05 09:11:05.457102+00:00
3,SPY,685.1600,+0.35%,"5,219,883",2026-03-05 09:11:05.457102+00:00



ALERTS:
 - [RSI] MSFT RSI 20.6 oversold (<= 30)
